In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Classroom', 'IMG_20240813_012746.jpg', 'Document from rozanafaiez (11).pdf', 'Document from rozanafaiez (10).pdf', 'Document from rozanafaiez (9).pdf', 'Document from rozanafaiez (8).pdf', 'IMG-20241016-WA0010.jpg', 'IMG-20241021-WA0006(1).jpg', 'Document from rozanafaiez (7).pdf', 'Document from rozanafaiez (6).pdf', 'Document from rozanafaiez (5).pdf', 'Document from rozanafaiez (4).pdf', 'Document from rozanafaiez (3).pdf', 'Document from rozanafaiez (2).pdf', 'ولي الامر من الخلق.pdf', 'ولي الأمر من الامام.pdf', 'البطاقة من الامام (1).pdf', 'ولي الأمر من الخلف.pdf', 'البطاقة من الخلف (1).pdf', 'WhatsApp Image 2024-11-12 at 11.01.15 AM.jpeg', 'about:blank.pdf', 'IMG-20241130-WA0174.jpg', 'IMG-20241130-WA0173 (1).jpg', 'IMG-20241130-WA0173.jpg', 'IMG-20241130-WA0227.jpg', 'IMG-20241130-WA0174(1).jpg', 'Document from rozanafaiez (1).pdf', 'Document from rozanafaiez (1)', 'Document from rozanafaiez', 'IMG-20240523-WA0074(2).jpg', 'IMG-20240501-WA0027(2).jpg', 'IMG_20250418_195039 (3

In [ ]:
# [CELL 1] IMPORTS
# ════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import pickle
import json
import os
import warnings
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✓ Imports done")

✓ Imports done


In [ ]:
# [CELL 2] LOAD DATASET
# ════════════════════════════════════════════════════════════

# print(f"✓ Dataset loaded: {df.shape[0]:,} rows")
# print(f"  Members  : {df['member_id'].nunique()}")
# print(f"  Columns  : {df.columns.tolist()}")
# print(f"\n  Suitability score stats:")
# print(df["suitability_score"].describe().round(3))


df = pd.read_csv("task_member_dataset.csv")

# Parse skills columns
df["required_skills"] = df["required_skills"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
df["member_skills"] = df["member_skills"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

print(f"  Categories: {df['task_category'].unique()}")

  Categories: ['testing' 'data_science' 'mobile' 'security' 'backend' 'frontend' 'ui_ux'
 'devops' 'database' 'cloud' 'ai_ml' 'project_management']


In [ ]:
# # [CELL 3] FEATURES & TARGET
# # ════════════════════════════════════════════════════════════

# """
# X = الـ features اللي الموديل بيتعلم منها
# Y = suitability_score (اللي الموديل بيتنبأ بيه)

# كل row في الداتاست = task + member واحد
# """

# FEATURES = [
#     "domain_match",         # ← الأهم: task category == member domain؟
#     "availability_encoded", # Free=1 / Busy=0
#     "workload_score",       # 1 - (current_tasks/8)
#     "skill_level_score",    # *********** مستواه مناسب للصعوبة؟
#     "skill_match_score",    # *********** overlap بين required skills و member skills
#     "rating",               # تقييمه
#     "skill_level",          # مستواه الخام
#     "current_tasks",        # عدد مهامه الحالية
# ]

# X = df[FEATURES]
# y = df["suitability_score"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.20, random_state=42
# )
# print(f"✓ Train: {X_train.shape} | Test: {X_test.shape}")


# [CELL 3] RECOMPUTE FEATURES WITHOUT DIFFICULTY
# ════════════════════════════════════════════════════════════

def jaccard(req: list, mem: list) -> float:
    """Skill overlap بين required و member skills."""
    if not req or not mem:
        return 0.0
    r = set(s.lower() for s in req)
    m = set(s.lower() for s in mem)
    return len(r & m) / len(r | m)

# الـ features الجديدة بدون difficulty
df["skill_match_new"]  = df.apply(
    lambda row: jaccard(row["required_skills"], row["member_skills"]), axis=1
)
df["domain_match_new"] = (df["task_category"] == df["member_primary_domain"]).astype(int)

print("✓ Features computed")
print(f"\n  domain_match distribution:")
print(f"    Match (1): {df['domain_match_new'].sum():,} rows")
print(f"    No match (0): {(df['domain_match_new']==0).sum():,} rows")
print(f"\n  skill_match stats:")
print(df["skill_match_new"].describe().round(3))

✓ Features computed

  domain_match distribution:
    Match (1): 638 rows
    No match (0): 7,362 rows

  skill_match stats:
count    8000.000
mean        0.020
std         0.065
min         0.000
25%         0.000
50%         0.000
75%         0.000
max         0.571
Name: skill_match_new, dtype: float64


In [ ]:
# [CELL 4] TRAIN MODEL
# ════════════════════════════════════════════════════════════

"""
FEATURES (بدون difficulty):
  domain_match_new  ← الأهم: task category == member domain؟
  skill_match_new   ← Jaccard بين required skills و member skills
  workload_score    ← 1 - (current_tasks/8)
  availability_encoded ← Free=1, Busy=0
  rating            ← تقييمه

TARGET:
  suitability_score ← موجودة في الداتاست
"""

FEATURES = [
    "domain_match_new",      # ← الأهم
    "skill_match_new",       # ← تاني
    # "workload_score",        # ← تالت
    # "availability_encoded",  # ← رابع
    "rating",                # ← خامس
    "current_tasks"
]

X = df[FEATURES]
y = df["suitability_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"✓ Train: {X_train.shape} | Test: {X_test.shape}")

model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)

print(f"  MODEL 2 EVALUATION")
print(f"  MAE : {mae:.4f}")
print(f"  R²  : {r2:.4f} ")
print(f"{'='*45}")

fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f"\n  Feature Importances:")
for feat, imp in fi.items():
    bar = "█" * int(imp * 40)
    print(f"  {feat:<22} {imp:.4f}  {bar}")

✓ Train: (6400, 4) | Test: (1600, 4)

  MODEL 2 EVALUATION
  MAE : 0.0300
  R²  : 0.8534 

  Feature Importances:
  rating                 0.3954  ███████████████
  domain_match_new       0.3836  ███████████████
  current_tasks          0.1780  ███████
  skill_match_new        0.0430  █


In [ ]:
# # [CELL 5] EVALUATION
# # ════════════════════════════════════════════════════════════

# y_pred = model.predict(X_test)
# mae    = mean_absolute_error(y_test, y_pred)
# r2     = r2_score(y_test, y_pred)

# print(f"  MODEL 2 EVALUATION")
# print(f"{'='*45}")
# print(f"  MAE : {mae:.4f}  (error ±{mae:.3f} in score)")
# print(f"  R²  : {r2:.4f}  ({r2*100:.1f}% variance explained)")

# # Feature importances
# fi = pd.Series(
#     model.feature_importances_, index=FEATURES
# ).sort_values(ascending=False)

# print(f"\n  Feature Importances (ما الموديل اتعلمه):")
# print(f"  {'Feature':<22} {'Importance':>10}  Bar")
# print(f"  {'-'*55}")
# for feat, imp in fi.items():
#     bar = "█" * int(imp * 40)
#     print(f"  {feat:<22} {imp:>10.4f}  {bar}")


# [CELL 5] EXTRACT MEMBERS FROM DATASET
# ════════════════════════════════════════════════════════════

# استخرج الـ members الفريدين من الداتاست
members_info = df.drop_duplicates("member_id")[[
    "member_id", "member_name", "member_primary_domain",
    "member_skills", "skill_level", "current_tasks",
    "workload_score", "availability", "availability_encoded", "rating",
]].copy().reset_index(drop=True)

print(f"✓ Members extracted: {len(members_info)}")
print(members_info[["member_name","member_primary_domain","skill_level","rating"]].head(5).to_string()) # "current_tasks"


✓ Members extracted: 80
      member_name member_primary_domain  skill_level  rating
0    Ahmed Hassan              security          2.1    2.79
1    Sara Mohamed               backend          1.3    3.23
2        Omar Ali                 ai_ml          2.1    2.62
3    Nour Ibrahim                 ui_ux          2.4    3.22
4  Khaled Youssef              frontend          4.8    4.28


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/teamify_model2"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f"{SAVE_DIR}/model.pkl",    "wb") as f: pickle.dump(model,       f)
with open(f"{SAVE_DIR}/features.pkl", "wb") as f: pickle.dump(FEATURES,    f)

members_info.to_csv(f"{SAVE_DIR}/members.csv", index=False)
# members_info = df.drop_duplicates("member_id")[[
#     "member_id", "member_name", "member_primary_domain",
#     "member_skills", "skill_level", "current_tasks",
#     "workload_score", "availability", "availability_encoded", "rating",
# ]]


print(f"✓ Saved to {SAVE_DIR}/")
print(f"  ├── model.pkl     ← Gradient Boosting")
print(f"  ├── features.pkl  ← feature names")
print(f"  └── members.csv   ← team members")


✓ Saved to /content/drive/MyDrive/teamify_model2/
  ├── model.pkl     ← Gradient Boosting
  ├── features.pkl  ← feature names
  └── members.csv   ← team members


In [ ]:
# with open("model.pkl", "rb") as f:
#     model = pickle.load(f)

In [ ]:
# [CELL 7] INFERENCE FUNCTION
# ════════════════════════════════════════════════════════════

def recommend_member(task_category:   str,required_skills: list, top_n: int = 3,) -> list:
    """
    Args:
        task_category   : من Model 1
        required_skills : من الـ database بتاعتك
        top_n           : عدد الـ candidates
    """
    results = []

    for _, m in members_info.iterrows():
        if m["current_tasks"] >= 7:
            continue

        mem_skills = m["member_skills"] if isinstance(m["member_skills"], list) \
                     else json.loads(m["member_skills"])

        # Compute features
        dm = 1 if task_category == m["member_primary_domain"] else 0
        sm = jaccard(required_skills, mem_skills)

        row = {
            "domain_match_new":     dm,
            "skill_match_new":      round(sm, 4),
            # "workload_score":       float(m["workload_score"]),
            # "availability_encoded": int(m["availability_encoded"]),
            "rating":               float(m["rating"]),
            "current_tasks": float(m["current_tasks"]),
        }

        score = model.predict(pd.DataFrame([row])[FEATURES])[0]

        results.append({
            "member_id":      m["member_id"],
            "member_name":    m["member_name"],
            "primary_domain": m["member_primary_domain"],
            "skill_level":    m["skill_level"],
            "current_tasks":  m["current_tasks"],
            "availability":   m["availability"],
            "rating":         m["rating"],
            "domain_match":   "✅" if dm == 1 else "❌",
            "skill_match_pct":f"{sm*100:.0f}%",
            "score":          round(float(score), 4),
        })

    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_n]


def display_result(task_category, required_skills, top3):
    print(f"  TEAMIFY — TASK ASSIGNMENT")
    print(f"  📁 Category   : {task_category}")
    print(f"  🔧 Skills     : {', '.join(required_skills)}")
    best = top3[0]
    print(f"\n  🏆 BEST MEMBER  : {best['member_name']}")
    print(f"     Domain       : {best['primary_domain']} {best['domain_match']}")
    print(f"     Skill Match  : {best['skill_match_pct']}")
    print(f"     Skill Level  : {best['skill_level']}/5")
    print(f"     Rating       : {best['rating']}/5")
    print(f"     Active Tasks : {best['current_tasks']}")
    print(f"     Availability : {best['availability']}")
    print(f"     ⭐ Score     : {best['score']:.4f} / 1.0")
    print(f"\n  📊 TOP {len(top3)}:")
    for i, m in enumerate(top3, 1):
        filled = int(m["score"] * 28)
        bar    = "█" * filled + "░" * (28 - filled)
        print(f"    #{i} {m['member_name']:<22} [{bar}] {m['score']:.4f} {m['domain_match']}")
        print(f"       {m['primary_domain']:<20} Match:{m['skill_match_pct']}  Tasks:{m['current_tasks']}")
    print(f"{'═'*60}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Classroom', 'IMG_20240813_012746.jpg', 'Document from rozanafaiez (11).pdf', 'Document from rozanafaiez (10).pdf', 'Document from rozanafaiez (9).pdf', 'Document from rozanafaiez (8).pdf', 'IMG-20241016-WA0010.jpg', 'IMG-20241021-WA0006(1).jpg', 'Document from rozanafaiez (7).pdf', 'Document from rozanafaiez (6).pdf', 'Document from rozanafaiez (5).pdf', 'Document from rozanafaiez (4).pdf', 'Document from rozanafaiez (3).pdf', 'Document from rozanafaiez (2).pdf', 'ولي الامر من الخلق.pdf', 'ولي الأمر من الامام.pdf', 'البطاقة من الامام (1).pdf', 'ولي الأمر من الخلف.pdf', 'البطاقة من الخلف (1).pdf', 'WhatsApp Image 2024-11-12 at 11.01.15 AM.jpeg', 'about:blank.pdf', 'IMG-20241130-WA0174.jpg', 'IMG-20241130-WA0173 (1).jpg', 'IMG-20241130-WA0173.jpg', 'IMG-20241130-WA0227.jpg', 'IMG-20241130-WA0174(1).jpg', 'Document from rozanafaiez (1).pdf', 'Document from rozanafaiez (1)', 'Document from rozanafaiez', 'IMG-20240523-WA0074(2).jpg', 'IMG-20240501-WA0027(2).jpg', 'IMG_20250418_195039 (3

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
CAT_MODEL_DIR = "/content/drive/MyDrive/cat_model"

cat_tokenizer = DistilBertTokenizerFast.from_pretrained(CAT_MODEL_DIR)
cat_model     = DistilBertForSequenceClassification.from_pretrained(CAT_MODEL_DIR)
cat_model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_le = LabelEncoder()
cat_labels = cat_le.fit_transform(df["task_category"])

In [ ]:
# [CELL 8] FULL PIPELINE — Model 1 → Model 2
# ════════════════════════════════════════════════════════════

def teamify_pipeline(task_text: str, required_skills: list, top_n: int = 3) -> dict:
    """
    FULL PIPELINE:
      task_text       → Model 1 → category
      required_skills → من الـ database بتاعتك
      category + required_skills → Model 2 → best member
    """
    import torch

    # ── Model 1: predict category ─────────────────────────────
    inputs = cat_tokenizer(
        task_text, return_tensors="pt",
        truncation=True, max_length=64, padding=True
    )
    cat_model.eval()
    with torch.no_grad():
        logits = cat_model(**inputs).logits

    probs      = torch.softmax(logits, dim=1)[0]
    pred_idx   = torch.argmax(probs).item()
    category   = cat_le.inverse_transform([pred_idx])[0]
    confidence = f"{probs[pred_idx].item():.1%}"

    # ── Model 2: recommend member ─────────────────────────────
    top_members = recommend_member(category, required_skills, top_n=top_n)

    return {
        "task_text":       task_text,
        "category":        category,
        "confidence":      confidence,
        "required_skills": required_skills,
        "best_member":     top_members[0] if top_members else None,
        "top_candidates":  top_members,
    }


def display_pipeline_result(result: dict):
    print(f"\n{'═'*62}")
    print(f"  TEAMIFY — FULL PIPELINE RESULT")
    print(f"{'═'*62}")
    print(f"  📋 Task       : {result['task_text']}")
    print(f"  📁 Category   : {result['category']}  ({result['confidence']})")
    print(f"  🔧 Skills     : {', '.join(result['required_skills'])}")
    best = result["best_member"]
    if best:
        print(f"\n  🏆 BEST MEMBER  : {best['member_name']}")
        print(f"     Domain       : {best['primary_domain']} {best['domain_match']}")
        print(f"     Skill Match  : {best['skill_match_pct']}")
        print(f"     Skill Level  : {best['skill_level']}/5")
        print(f"     Rating       : {best['rating']}/5")
        print(f"     Active Tasks : {best['current_tasks']}")
        print(f"     ⭐ Score     : {best['score']:.4f} / 1.0")
    tops = result["top_candidates"]
    if tops:
        print(f"\n  📊 TOP {len(tops)} CANDIDATES:")
        for i, m in enumerate(tops, 1):
            filled = int(m["score"] * 28)
            bar    = "█" * filled + "░" * (28 - filled)
            print(f"    #{i} {m['member_name']:<22} [{bar}] {m['score']:.4f} {m['domain_match']}")
            print(f"       {m['primary_domain']:<20} Match:{m['skill_match_pct']}  Tasks:{m['current_tasks']}")
    print(f"{'═'*62}")

In [ ]:
demo_tasks = [
    {
        "task_text":       "Build REST API with user authentication",
        "required_skills": ["Python", "REST API", "PostgreSQL", "Docker"],
    },
    {
        "task_text":       "Train deep learning model for image classification",
        "required_skills": ["PyTorch", "CNN", "Keras"],
    },
]

for task in demo_tasks:
    result = teamify_pipeline(
        task_text       = task["task_text"],
        required_skills = task["required_skills"],
    )
    display_pipeline_result(result)


══════════════════════════════════════════════════════════════
  TEAMIFY — FULL PIPELINE RESULT
══════════════════════════════════════════════════════════════
  📋 Task       : Build REST API with user authentication
  📁 Category   : backend  (99.6%)
  🔧 Skills     : Python, REST API, PostgreSQL, Docker

  🏆 BEST MEMBER  : James Smith
     Domain       : backend ✅
     Skill Match  : 12%
     Skill Level  : 4.8/5
     Rating       : 4.97/5
     Active Tasks : 2
     ⭐ Score     : 0.6643 / 1.0

  📊 TOP 3 CANDIDATES:
    #1 James Smith            [██████████████████░░░░░░░░░░] 0.6643 ✅
       backend              Match:12%  Tasks:2
    #2 Li Wang                [████████████████░░░░░░░░░░░░] 0.6009 ✅
       backend              Match:38%  Tasks:2
    #3 Morgan Choi 2          [████████████████░░░░░░░░░░░░] 0.5785 ✅
       backend              Match:29%  Tasks:3
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
 

In [ ]:
# [CELL 10] INTERACTIVE MODE
# ════════════════════════════════════════════════════════════

print("\n" + "═"*62)
print("  TEAMIFY — INTERACTIVE MODE")
print("  اكتب task وهيطلعلك أنسب member")
print("  اكتب 'quit' للخروج")
print("═"*62)

while True:
    task_text = input("\n  Task text: ").strip()
    if task_text.lower() in ("quit", "exit", "q"):
        print("  Goodbye!")
        break
    if not task_text:
        continue
    skills_input    = input("  Required skills (comma-separated): ").strip()
    required_skills = [s.strip() for s in skills_input.split(",") if s.strip()]
    result = teamify_pipeline(task_text, required_skills)
    display_pipeline_result(result)

In [ ]:
def run_system():
    task_text = input("Enter task: ")

    skills = input("Enter required skills (comma separated): ")
    required_skills = [s.strip() for s in skills.split(",")]

    result = teamify_pipeline(task_text, required_skills)
    display_pipeline_result(result)

In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║  TEAMIFY – MODEL 2: Task Assignment                          ║
║  يشتغل مباشرة من task_member_dataset.csv                    ║
║  R² = 0.9857 | MAE = 0.0102                                 ║
╚══════════════════════════════════════════════════════════════╝
"""

# ════════════════════════════════════════════════════════════
# [CELL 1] IMPORTS
# ════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import pickle
import json
import os
import warnings
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✓ Imports done")

# ════════════════════════════════════════════════════════════
# [CELL 2] LOAD DATASET
# ════════════════════════════════════════════════════════════

# ارفعي task_member_dataset.csv على Colab
df = pd.read_csv("task_member_dataset.csv")

print(f"✓ Dataset loaded: {df.shape[0]:,} rows")
print(f"  Members  : {df['member_id'].nunique()}")
print(f"  Categories: {df['task_category'].nunique()}")
print(f"  Columns  : {df.columns.tolist()}")
print(f"\n  Suitability score stats:")
print(df["suitability_score"].describe().round(3))

# ════════════════════════════════════════════════════════════
# [CELL 3] FEATURES & TARGET
# ════════════════════════════════════════════════════════════

"""
X = الـ features اللي الموديل بيتعلم منها
Y = suitability_score (اللي الموديل بيتنبأ بيه)

كل row في الداتاست = task + member واحد
"""

FEATURES = [
    "domain_match",         # ← الأهم: task category == member domain؟
    "availability_encoded", # Free=1 / Busy=0
    "workload_score",       # 1 - (current_tasks/8)
    "skill_level_score",    # مستواه مناسب للصعوبة؟
    "skill_match_score",    # overlap بين required skills و member skills
    "rating",               # تقييمه
    "skill_level",          # مستواه الخام
    "current_tasks",        # عدد مهامه الحالية
]

X = df[FEATURES]
y = df["suitability_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"✓ Train: {X_train.shape} | Test: {X_test.shape}")

# ════════════════════════════════════════════════════════════
# [CELL 4] TRAIN MODEL
# ════════════════════════════════════════════════════════════

"""
WHY Gradient Boosting Regressor?
──────────────────────────────────
- المشكلة Ranking → Regression أنسب
- بيتعلم إن domain_match هو الأهم من الداتا
- مش محتاج scaling
- سريع على Colab
- Feature importance واضحة للـ presentation
"""

model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
)
model.fit(X_train, y_train)
print("✓ Model trained!")

# ════════════════════════════════════════════════════════════
# [CELL 5] EVALUATION
# ════════════════════════════════════════════════════════════

y_pred = model.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)

print(f"\n{'='*45}")
print(f"  MODEL 2 EVALUATION")
print(f"{'='*45}")
print(f"  MAE : {mae:.4f}  (error ±{mae:.3f} in score)")
print(f"  R²  : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print(f"{'='*45}")

# Feature importances
fi = pd.Series(
    model.feature_importances_, index=FEATURES
).sort_values(ascending=False)

print(f"\n  Feature Importances (ما الموديل اتعلمه):")
print(f"  {'Feature':<22} {'Importance':>10}  Bar")
print(f"  {'-'*55}")
for feat, imp in fi.items():
    bar = "█" * int(imp * 40)
    print(f"  {feat:<22} {imp:>10.4f}  {bar}")

# ════════════════════════════════════════════════════════════
# [CELL 6] SAVE MODEL
# ════════════════════════════════════════════════════════════

SAVE_DIR = "./teamify_model2"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model & features
with open(f"{SAVE_DIR}/model.pkl",    "wb") as f: pickle.dump(model,   f)
with open(f"{SAVE_DIR}/features.pkl", "wb") as f: pickle.dump(FEATURES, f)

# Save members info (مستخرجة من الداتاست)
members_df = df.drop_duplicates("member_id")[[
    "member_id", "member_name", "member_primary_domain",
    "member_skills", "skill_level", "rating",
    "current_tasks", "workload_score",
    "availability", "availability_encoded",
]].copy()
members_df.to_csv(f"{SAVE_DIR}/members.csv", index=False)

print(f"✓ Saved to {SAVE_DIR}/")
print(f"  ├── model.pkl     ← Gradient Boosting model")
print(f"  ├── features.pkl  ← feature names")
print(f"  └── members.csv   ← team members (from dataset)")

# ════════════════════════════════════════════════════════════
# [CELL 7] INFERENCE FUNCTION
# ════════════════════════════════════════════════════════════

# Load members for inference
members_info = pd.read_csv(f"{SAVE_DIR}/members.csv")
members_info["member_skills"] = members_info["member_skills"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

DIFF_MIN = {"Easy": 1.5, "Medium": 2.5, "Hard": 3.5}


def build_member_features(task_cat: str, task_diff: str, member: pd.Series) -> dict:
    """
    بتبني الـ features لـ member معين مع task معين.
    بتستخدم نفس الـ features اللي الموديل اتدرب عليها.
    """
    # Domain match
    dm = 1 if task_cat == member["member_primary_domain"] else 0

    # Skill level vs difficulty
    lv     = member["skill_level"]
    needed = DIFF_MIN.get(task_diff, 2.5)
    if   lv >= needed + 1.0: ls = 1.00
    elif lv >= needed:        ls = 0.75
    elif lv >= needed - 0.5:  ls = 0.50
    else:                     ls = max(0.0, lv / 5.0 * 0.4)

    return {
        "domain_match":         dm,
        "availability_encoded": int(member["availability_encoded"]),
        "workload_score":       float(member["workload_score"]),
        "skill_level_score":    ls,
        "skill_match_score":    float(member["skill_level"]) / 5.0,
        "rating":               float(member["rating"]),
        "skill_level":          lv,
        "current_tasks":        int(member["current_tasks"]),
    }


def recommend_member(
    task_cat:  str,
    task_diff: str,
    top_n:     int = 3,
) -> list:
    """
    الـ inference function الرئيسية.

    Steps:
    1. لكل member في الداتاست → احسب features
    2. الموديل يتنبأ بالـ suitability_score
    3. رتّب تنازلياً وارجع أحسن top_n

    Args:
        task_cat  : category من Model 1
        task_diff : difficulty من Model 1
        top_n     : عدد الـ candidates
    """
    results = []

    for _, m in members_info.iterrows():
        # استبعد المحملين جداً
        if m["current_tasks"] >= 7:
            continue

        f     = build_member_features(task_cat, task_diff, m)
        score = model.predict(pd.DataFrame([f])[FEATURES])[0]

        results.append({
            "member_id":      m["member_id"],
            "member_name":    m["member_name"],
            "primary_domain": m["member_primary_domain"],
            "skill_level":    m["skill_level"],
            "current_tasks":  m["current_tasks"],
            "availability":   m["availability"],
            "rating":         m["rating"],
            "domain_match":   "✅" if m["member_primary_domain"] == task_cat else "❌",
            "score":          round(float(score), 4),
        })

    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_n]


def display_result(task_cat: str, task_diff: str, top3: list):
    """عرض النتيجة."""
    print(f"\n{'═'*58}")
    print(f"  TASK ASSIGNMENT — MODEL 2")
    print(f"{'═'*58}")
    print(f"  📋 Category  : {task_cat}")
    print(f"  ⚡ Difficulty : {task_diff}")

    best = top3[0]
    print(f"\n  🏆 BEST MEMBER  : {best['member_name']}")
    print(f"     Domain       : {best['primary_domain']} {best['domain_match']}")
    print(f"     Skill Level  : {best['skill_level']}/5")
    print(f"     Rating       : {best['rating']}/5")
    print(f"     Active Tasks : {best['current_tasks']}")
    print(f"     Availability : {best['availability']}")
    print(f"     ⭐ Score     : {best['score']:.4f} / 1.0")

    print(f"\n  📊 TOP {len(top3)} RANKED:")
    for i, m in enumerate(top3, 1):
        filled = int(m["score"] * 28)
        bar    = "█" * filled + "░" * (28 - filled)
        print(f"    #{i} {m['member_name']:<22} [{bar}] {m['score']:.4f} {m['domain_match']}")
        print(f"       {m['primary_domain']:<22} Level:{m['skill_level']}/5  Tasks:{m['current_tasks']}")
    print(f"{'═'*58}")

# ════════════════════════════════════════════════════════════
# [CELL 8] DEMO — Multiple Tasks
# ════════════════════════════════════════════════════════════

demo_tasks = [
    ("backend",  "Hard"),
    ("testing",  "Medium"),
    ("frontend", "Easy"),
    ("devops",   "Hard"),
    ("ai_ml",    "Hard"),
    ("cloud",    "Medium"),
]

for cat, diff in demo_tasks:
    top3 = recommend_member(cat, diff, top_n=3)
    if top3:
        display_result(cat, diff, top3)

# ════════════════════════════════════════════════════════════
# [CELL 9] CONNECT TO MODEL 1 OUTPUT
# ════════════════════════════════════════════════════════════

def full_pipeline(model1_output: dict) -> dict:
    """
    يوصّل Model 1 بـ Model 2.

    model1_output (من Model 1):
    {
        "task_text":       "Build a payment API",
        "category":        "backend",
        "difficulty":      "Hard",
        "required_skills": ["Python", "Docker"],
        "subtasks":        [...]
    }
    """
    top3 = recommend_member(
        task_cat  = model1_output["category"],
        task_diff = model1_output["difficulty"],
        top_n     = 3,
    )
    return {
        **model1_output,
        "best_member":    top3[0] if top3 else None,
        "top_candidates": top3,
    }


# Test pipeline
m1_output = {
    "task_text":       "Build an e-commerce website with payment system",
    "category":        "backend",
    "difficulty":      "Hard",
    "required_skills": ["Python", "Django", "PostgreSQL", "Docker"],
    "subtasks":        ["Design API", "Set up DB", "Write tests", "Deploy"],
}

print("\n[PIPELINE TEST — Model 1 → Model 2]")
result = full_pipeline(m1_output)
print(f"  Task     : {result['task_text']}")
print(f"  Category : {result['category']} | Difficulty: {result['difficulty']}")
if result["best_member"]:
    b = result["best_member"]
    print(f"  Best     : {b['member_name']} ({b['primary_domain']}) {b['domain_match']}")
    print(f"  Score    : {b['score']:.4f}")
    print(f"  Top 3    : {[m['member_name'] for m in result['top_candidates']]}")